# Agent-Based Wound Healing Simulation
### Section 1: Visualizing Scenarios with Reinfection and Healer Boost (Natural Immunity)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap

# -- Dark-mode palette --
_BG        = "#0d1117"
_AXES_BG   = "#161b22"
_EDGE      = "#30363d"
_TEXT      = "#e6edf3"
_TICK      = "#8b949e"
_RED_HOT   = "#f85149"
_RED_WOUND = "#6e1b1b"
_GREEN_HLD = "#196127"
_BLUE_WALK = "#58a6ff"
_YELLOW_INF = "#f1e05a"
_CYAN_WALK  = "#ff00ff"

plt.rcParams.update({
    "figure.facecolor":  _BG, "axes.facecolor": _AXES_BG, "axes.edgecolor": _EDGE,
    "axes.labelcolor": _TEXT, "axes.titlecolor": _TEXT, "text.color": _TEXT,
    "xtick.color": _TICK, "ytick.color": _TICK, "grid.color": _EDGE,
    "legend.facecolor": _AXES_BG, "legend.edgecolor": _EDGE,
    "savefig.facecolor": _BG, "axes.titlepad": 10,
})

# -- Shared Configuration --
grid_size = 150
num_steps = 1500
heal_threshold = 3
snapshot_every = 10
n_spawn_points = 30
walkers_per_spawn = 10
rng = np.random.default_rng(42)

def generate_noisy_boundary_points(center_xy, base_radius, n_points):
    angles = np.linspace(0, 2 * np.pi, n_points, endpoint=True)
    harmonic = 1 + 0.1 * np.sin(4 * angles) + 0.18 * np.sin(3 * angles)
    radius = np.clip(base_radius * harmonic + rng.normal(0.0, 3.0, size=n_points), base_radius * 0.45, base_radius * 1.45)
    return np.column_stack([center_xy[0] + radius * np.cos(angles), center_xy[1] + radius * np.sin(angles)])

def polygon_to_mask(boundary_points_xy, size):
    yy, xx = np.mgrid[0:size, 0:size]
    all_pixels_xy = np.column_stack([xx.ravel(), yy.ravel()])
    return Path(boundary_points_xy).contains_points(all_pixels_xy).reshape(size, size)

def run_simulation(mask, start_cells_rc, n_steps, snapshot_every, healer_ratio, spread_chance, boost_chance=0.01):
    walkers = np.repeat(start_cells_rc, walkers_per_spawn, axis=0).astype(int)
    n_walkers = len(walkers)
    walker_roles = np.zeros(n_walkers, dtype=int)
    role_indices = rng.choice(n_walkers, int(n_walkers * healer_ratio), replace=False)
    walker_roles[role_indices] = 1
    
    visit_counts = np.zeros(mask.shape, dtype=np.int32)
    healed = np.zeros(mask.shape, dtype=bool)
    infected = np.zeros(mask.shape, dtype=bool)
    
    wound_idx = np.argwhere(mask)
    infected[tuple(wound_idx[rng.choice(len(wound_idx), int(len(wound_idx)*0.02))].T)] = True
    
    snapshots = []
    offsets = np.array([[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]])

    for step in range(n_steps):
        # 1. Spreading / Reinfection logic
        inf_r, inf_c = np.where(infected)
        for r, c in zip(inf_r, inf_c):
            if rng.random() < spread_chance:
                off = offsets[rng.integers(8)]
                nr, nc = r + off[0], c + off[1]
                if (0 <= nr < grid_size and 0 <= nc < grid_size and mask[nr, nc]):
                    infected[nr, nc] = True
                    healed[nr, nc] = False

        # 2. Healer Boost Logic (Healthy cells help neighbors)
        healed_r, healed_c = np.where(healed)
        for r, c in zip(healed_r, healed_c):
            off = offsets[rng.integers(8)]
            nr, nc = r + off[0], c + off[1]
            if (0 <= nr < grid_size and 0 <= nc < grid_size and infected[nr, nc]):
                if rng.random() < boost_chance:
                    infected[nr, nc] = False
                    healed[nr, nc] = True

        # 3. Walker movement and action logic
        for i in range(n_walkers):
            pos = walkers[i]
            drift_vec = rng.normal(0, 0.5, size=2)
            candidates = pos + offsets
            valid = (candidates[:,0] >= 0) & (candidates[:,0] < mask.shape[0]) & \
                    (candidates[:,1] >= 0) & (candidates[:,1] < mask.shape[1])
            candidates = candidates[valid]
            candidates = candidates[mask[candidates[:,0], candidates[:,1]]]
            if len(candidates) > 0:
                scores = (candidates - pos) @ drift_vec
                walkers[i] = candidates[np.argmax(scores)]
            
            new_r, new_c = walkers[i]
            if walker_roles[i] == 1: # Healer
                if infected[new_r, new_c]:
                    infected[new_r, new_c] = False
                    healed[new_r, new_c] = True
            else: # Regular
                if not infected[new_r, new_c]:
                    visit_counts[new_r, new_c] += 1
                    if visit_counts[new_r, new_c] >= heal_threshold:
                        healed[new_r, new_c] = True

        if step % snapshot_every == 0:
            snapshots.append({'step': step, 'infected': infected.copy(), 'healed': healed.copy(), 'walkers': walkers.copy(), 'roles': walker_roles.copy()})
    return snapshots

def save_gif(snapshots, mask, filename, title):
    cmap_inf = ListedColormap([_BG, _RED_WOUND, _YELLOW_INF, _GREEN_HLD])
    fig, ax = plt.subplots(figsize=(6, 6))
    def update(frame_idx):
        ax.clear()
        data = snapshots[frame_idx]
        state = np.zeros(mask.shape, dtype=np.uint8)
        state[mask] = 1
        state[data['infected']] = 2
        state[data['healed']] = 3
        ax.imshow(state, cmap=cmap_inf, origin="lower")
        healers = data['walkers'][data['roles'] == 1]
        regulars = data['walkers'][data['roles'] == 0]
        ax.scatter(regulars[:, 1], regulars[:, 0], c=_BLUE_WALK, s=8, alpha=0.6)
        ax.scatter(healers[:, 1], healers[:, 0], c=_CYAN_WALK, s=20, edgecolors="white", linewidths=0.3)
        h_pct = (data['healed'].sum() / mask.sum()) * 100
        ax.set_title(f"{title}\nStep {data['step']} | Healed: {h_pct:.1f}% | Infected: {data['infected'].sum()}")
        ax.axis("off")
    anim = FuncAnimation(fig, update, frames=len(snapshots), interval=50)
    anim.save(filename, writer=PillowWriter(fps=15))
    plt.close(fig)

# -- Initialize Geometry --
true_center_xy = np.array([grid_size * 0.52, grid_size * 0.50])
boundary_points = generate_noisy_boundary_points(true_center_xy, grid_size*0.3, 150)
wound_mask = polygon_to_mask(boundary_points, grid_size)
boundary_cells_rc = np.argwhere(wound_mask & ~((np.pad(wound_mask, 1)[:-2, 1:-1] & np.pad(wound_mask, 1)[2:, 1:-1] & np.pad(wound_mask, 1)[1:-1, :-2] & np.pad(wound_mask, 1)[1:-1, 2:])))
start_cells_rc = boundary_cells_rc[rng.choice(len(boundary_cells_rc), n_spawn_points, replace=False)]

# Running Scenario 1
print("Running Scenario 1: Infection Wins...")
snaps_win = run_simulation(wound_mask, start_cells_rc, num_steps, snapshot_every, healer_ratio=0.1, spread_chance=0.02, boost_chance=0.01)
save_gif(snaps_win, wound_mask, "infection__wins.gif", "Scenario 1: Infection Wins")

# Running Scenario 2
print("Running Scenario 2: Infection Killed...")
snaps_kill = run_simulation(wound_mask, start_cells_rc, num_steps, snapshot_every, healer_ratio=0.5, spread_chance=0.02, boost_chance=0.01)
save_gif(snaps_kill, wound_mask, "infection__killed.gif", "Scenario 2: Infection Killed")
print("GIF animations successfully saved to disk!")

### Section 2: Standalone Workforce Efficiency Evaluation

In [ ]:
# Configured variables for standalone processing
ratios_to_test = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
healed_results = []
healer_counts = []
total_walkers = n_spawn_points * walkers_per_spawn

print(f"Starting standalone experiment over {len(ratios_to_test)} scenarios...")
print(f"Total Walkers per test: {total_walkers}")

for ratio in ratios_to_test:
    results = run_simulation(
        mask=wound_mask, 
        start_cells_rc=start_cells_rc, 
        n_steps=1500, 
        snapshot_every=1500,
        healer_ratio=ratio, 
        spread_chance=0.02,
        boost_chance=0.01
    )
    
    final_snap = results[-1]
    total_wound_cells = wound_mask.sum()
    remaining_infected = final_snap['infected'].sum()
    
    percentage = (1 - (remaining_infected / total_wound_cells)) * 100
    healed_results.append(percentage)
    healer_counts.append(int(total_walkers * ratio))
    print(f"Ratio: {ratio:.1f} ({healer_counts[-1]} healers) -> {percentage:.1f}% Score")

# -- Plotting the Results --
plt.figure(figsize=(10, 6))
plt.plot(healer_counts, healed_results, marker='o', linestyle='-', color=_BLUE_WALK, linewidth=2, markersize=8)
plt.title("Healer Effectiveness: Population vs. Infection Control", fontsize=14)
plt.xlabel(f"Amount of Healer Walkers (out of {total_walkers} total)", fontsize=12)
plt.ylabel("Infection-Free Score (%)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3, color=_EDGE)
plt.ylim(-5, 105)
plt.axhline(y=100, color=_GREEN_HLD, linestyle=':', alpha=0.8, label="100% Infection-Free")
plt.fill_between(healer_counts, 0, 20, color=_RED_HOT, alpha=0.1, label="Infection Danger Zone")
plt.legend()
plt.tight_layout()
plt.savefig("healer_efficiency_plot.png")
plt.show()

### Section 3: Head-to-Head Efficiency Comparison (With vs Without Boost)

In [ ]:
ratios_for_comparison = np.linspace(0.0, 0.7, 8)
scores_no_boost = []
scores_with_boost = []
all_cells = wound_mask.sum()

print("Running Head-to-Head Evaluation Experiment...")
for ratio in ratios_for_comparison:
    # Evaluated without boosting context
    res_no = run_simulation(wound_mask, start_cells_rc, 500, 500, ratio, 0.04, boost_chance=0.0)
    inf_no = res_no[-1]['infected'].sum()
    scores_no_boost.append((1 - (inf_no / all_cells)) * 100)
    
    # Evaluated with active structural neighborhood boosting
    res_yes = run_simulation(wound_mask, start_cells_rc, 500, 500, ratio, 0.04, boost_chance=0.01)
    inf_yes = res_yes[-1]['infected'].sum()
    scores_with_boost.append((1 - (inf_yes / all_cells)) * 100)
    print(f"Completed processing for baseline Healer Workforce Ratio: {ratio:.2f}")

# -- Visualization Configuration --
plt.figure(figsize=(10, 6))
comparison_counts = ratios_for_comparison * total_walkers

plt.plot(comparison_counts, scores_no_boost, 'o--', color=_RED_HOT, label="Without Healthy Boost")
plt.plot(comparison_counts, scores_with_boost, 'o-', color=_GREEN_HLD, linewidth=2, label="With Healthy Boost (Natural Immunity)")

plt.title("Comparison: The Impact of Natural Immunity on Healer Efficiency", fontsize=14)
plt.xlabel("Amount of Healer Walkers", fontsize=12)
plt.ylabel("Infection-Free Score (%)", fontsize=12)
plt.grid(True, alpha=0.2, color=_TICK)
plt.ylim(-5, 105)
plt.legend()
plt.tight_layout()
plt.show()